## This script shows a rapid visualization of micro raman maps aquired with Renisaw microscopes
@utor: Marti Checa

Spring 2025

In [1]:
%matplotlib

Using matplotlib backend: Qt5Agg


In [3]:
import matplotlib.pyplot as plt
import numpy as np
from renishawWiRE import WDFReader
from matplotlib.widgets import Slider, TextBox
import matplotlib.image as mpimg

# Load the Spectra Data
filename = r"F:\ORNL\Experimental data\Renishaw\ngay bridges\new bridges/LowRess_map_x100_50%power_7s_3accomulations_532nm_1800_higherress_onAliveBridge.wdf"
reader = WDFReader(filename)
spectra = reader.spectra

# Load the Optical Image
img = mpimg.imread(reader.img, format="jpg")

# Get the sizes of the imported file
Xpix = spectra.shape[0]
Ypix = spectra.shape[1]
Raman_axis_size = spectra.shape[2]

# Calculate Mean Spectrum
meanspectra = spectra.mean(axis=(0, 1))
meanspectra = np.flipud(meanspectra)

# Create a figure with subplots (Optical Image on the left)
fig, (ax3, ax1, ax2) = plt.subplots(1, 3, figsize=(18, 6))  # Optical image to the left
plt.subplots_adjust(bottom=0.25)

# Display the Optical Image
ax3.imshow(np.rot90(img,k=-2))
ax3.set_title('Optical Image')
ax3.axis('off')  # Hide axes for the optical image

# Initial plot of Raman map and Mean Spectra
raman_shift_init = 0
raman_map_image = ax1.imshow(np.rot90(spectra[:, :, raman_shift_init].T,k=-1), cmap='plasma', aspect=(Xpix / Ypix))
ax1.set_title(f'Raman Intensity at shift {reader.xdata[-(raman_shift_init + 1)]:.2f}')
ax1.set_xlabel('X Pixel Index')
ax1.set_ylabel('Y Pixel Index')

mean_spectrum_line, = ax2.plot(np.flipud(reader.xdata), meanspectra, label='Mean Spectrum')
selected_pixel_line, = ax2.plot([], [], label='Selected Pixel Spectrum', linestyle='--', color='blue')  # Initialize selected pixel line
# Initialize a vertical line for the selected Raman shift
vertical_line = ax2.axvline(x=reader.xdata[-(raman_shift_init + 1)], color='red', linestyle='--', linewidth=2, label='Selected Shift')

ax2.set_title('Mean Spectra')
ax2.set_xlabel('Raman Shift')
ax2.set_ylabel('Raman Intensity')
ax2.legend(title='Coordinates', loc='upper right')

# Add a slider for Raman Shift
ax_slider = plt.axes([0.1, 0.15, 0.5, 0.03])
slider = Slider(ax_slider, 'Raman Shift', 0, Raman_axis_size - 1, valinit=raman_shift_init, valstep=1)

# Adjust the position and size of the text box
ax_textbox = plt.axes([0.8, 0.15, 0.15, 0.03])  # Move to the right and resize
text_box = TextBox(ax_textbox, 'Raman Shift Value', initial=str(reader.xdata[-(raman_shift_init + 1)]))

def update(val):
    shift_index = int(slider.val)
    corrected_index = Raman_axis_size - 1 - shift_index
    corrected_index_line = shift_index

    raman_map_image.set_array(np.rot90(spectra[:, :, corrected_index].T,k=-1))
    raman_map_image.set_clim(vmin=np.mean(spectra[:, :, corrected_index])-2*np.std(spectra[:, :, corrected_index]), 
                             vmax=np.mean(spectra[:, :, corrected_index])+2*np.std(spectra[:, :, corrected_index]))
    ax1.set_title(f'Raman Intensity at shift {reader.xdata[-(corrected_index_line + 1)]:.2f}')
    vertical_line.set_xdata(reader.xdata[-(corrected_index_line + 1)])
    text_box.set_val(str(reader.xdata[-(corrected_index_line + 1)]))
    
    fig.canvas.draw_idle()

# Connect the slider to the update function
slider.on_changed(update)

def on_click(event):
    # Check if the click is within the axes
    if event.inaxes == ax1:
        # Get the x and y pixel index from the click
        y_idx = int(event.xdata)  # Note: x corresponds to pixel coordinates for the rotated image
        x_idx = int(event.ydata)
        
        # Check bounds (optional, in case of a click outside the map)
        if 0 <= x_idx < Xpix and 0 <= y_idx < Ypix:
            # Get the spectrum for the clicked pixel
            selected_spectrum = spectra[x_idx, spectra.shape[1]-y_idx, :]

            # Update the selected pixel line
            selected_pixel_line.set_data(np.flipud(reader.xdata), np.flipud(selected_spectrum))

            # Update the legend with the selected pixel coordinates
            ax2.legend(title=f'Selected Pixel: ({y_idx}, {x_idx})', loc='upper right')

            # Update axes limits and refresh display
            #ax2.relim()  # Recalculate limits
            #ax2.autoscale_view()  # Autoscale
            fig.canvas.draw_idle()  # Redraw the figure

# Connect the click event to the on_click function
cid = fig.canvas.mpl_connect('button_press_event', on_click)

plt.show()